In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn import linear_model
from sklearn import model_selection
# from sklearn import metrics
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    r2_score,
    classification_report
)
from sklearn import preprocessing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import KeyedVectors
# from gensim.models.fasttext import load_facebook_model
import gensim.downloader as api
import re
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm


TEST_SIZE = 0.2
RANDOM_SEED = 42

In [13]:
tokens = pd.read_csv('../data/comments_parsed.csv')

In [22]:
comments = pd.read_csv("../data/toxic_comments_with_morph.csv").drop(columns='Unnamed: 0')
comments = comments.drop(columns=['cleaned_comment', 'comment_without_punct', 'swearing.1'])

In [23]:
comments.columns, tokens.columns

(Index(['label', 'comment', 'length_sym', 'length_words', 'av_word_len',
        'is_normal', 'swearing', 'has_positive_emoji', 'has_negative_emoji',
        'has_obscene_emoji', 'imperative', '2nd_prsn_count', '3nd_prsn_count',
        'adj_count', 'vocative', 'expr_punct'],
       dtype='object'),
 Index(['tokens', 'lemma_pos', 'feats'], dtype='object'))

In [ ]:
df = pd.concat([comments, tokens], axis=1)

In [27]:
df

,label,comment,length_sym,length_words,av_word_len,is_normal,swearing,has_positive_emoji,has_negative_emoji,has_obscene_emoji,imperative,2nd_prsn_count,3nd_prsn_count,adj_count,vocative,expr_punct,tokens,lemma_pos,feats
0,INSULT,скотина! что сказать,20.0,3.0,5.666667,False,False,False,False,False,False,0.0,0.0,0.0,True,False,NaN,NaN,NaN
1,NORMAL,я сегодня проезжала по рабочей и между домами ...,180.0,28.0,5.214286,True,False,False,False,False,False,0.0,0.0,3.0,False,False,NaN,NaN,NaN
2,NORMAL,очередной лохотрон. зачем придумывать очередно...,379.0,54.0,5.777778,True,False,False,False,False,False,0.0,0.0,8.0,False,True,NaN,NaN,NaN
3,NORMAL,"ретро дежавю ... сложно понять чужое сердце , ...",72.0,10.0,5.700000,True,False,False,False,False,False,0.0,0.0,0.0,False,True,NaN,NaN,NaN
4,NORMAL,а когда мы статус агрогородка получили?,39.0,6.0,5.500000,True,False,False,False,False,False,0.0,0.0,0.0,False,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,правильно\tвсё\tпять,правильно_ADV\tвесь_DET\tпять_NUM,Degree=Pos\tCase=Nom|Gender=Neut|Number=Sing|P...
248277,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ёбанные\tнубы\tзаходите\tсервер\tник\t_creepro...,ебанный_ADJ\tнуб_NOUN\tзаходить_VERB\tсервер_N...,Case=Nom|Degree=Pos|Number=Plur\tAnimacy=Inan|...
248278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,наверное\tрекорд\tгоду\tучилище\tкоренной\tзуб...,наверное_ADV\tрекорд_NOUN\tгод_NOUN\tучилище_N...,Degree=Pos\tAnimacy=Inan|Case=Nom|Gender=Masc|...
248279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,спасибо\tвсем\tбольшое,спасибо_NOUN\tвсе_PRON\tбольшой_ADJ,Animacy=Inan|Case=Nom|Gender=Neut|Number=Sing\...
